In [ ]:
import numpy as np
import scipy.signal
import tensorstore as ts
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
import asyncio
# from tqdm.asyncio import tqdm_asyncio
from mpl_toolkits.axes_grid1 import ImageGrid

In [ ]:
# Pull data from google storage since the raw data takes a while to download
gs_uri = "gs://zapbench-release/volumes/20240930"
# gs_uri = "file:///groups/saalfeld/saalfeldlab/zapbench-release/volumes/20240930"
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/traces'
}).result()
# Check that we have a time stamp for each neuronal trace
# assert ds.shape[0] == timestamps_ms.shape[0]

In [ ]:
# Load segmentation
ds = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/segmentation'
}).result()
segmentation = ds.read().result()

In [ ]:
# process segmentation to build a map between voxels and cell id's

xi, yi, zi = np.where(segmentation > 0)


# get to zero based indexing
cell_id_flat = segmentation[xi, yi, zi].astype(np.uint64)
cell_id_flat -= 1


In [ ]:

# ============================================================================
# FLOW FIELDS SCHEMA
# ============================================================================
# Shape: [3, 36, 83, 128, 7879] with dimensions ["fc", "fz", "fy", "fx", "t"]
#
# The flow field stores DISPLACEMENTS (offsets) from aligned space to raw space:
#   raw_x = aligned_x + flow_fields[0, gz, gy, gx, t]
#   raw_y = aligned_y + flow_fields[1, gz, gy, gx, t]
#   raw_z = aligned_z + flow_fields[2, gz, gy, gx, t]
#
# Grid strides (aligned space -> flow field grid):
#   - stride_x = 2048 / 128 = 16
#   - stride_y = 1328 / 83  = 16
#   - stride_z = 72 / 36    = 2
# ============================================================================

# Volume dimensions
SIZE_X, SIZE_Y, SIZE_Z, SIZE_T = 2048, 1328, 72, 7879

# Flow field grid strides (aligned space pixels per grid point)
STRIDE_X = 16
STRIDE_Y = 16
STRIDE_Z = 2
ds_flow = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'{gs_uri}/flow_fields'
}).result()

# this is the raw data from the microscope
ds_raw = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/raw/'
}).result()

In [ ]:
gx = xi // STRIDE_X
gy = yi // STRIDE_Y
gz = zi // STRIDE_Z
T = 1000
offset = ds_flow[:, gz, gy, gx, T].read().result()
ioffset = np.round(offset / np.array([1, 1, 4])[:, np.newaxis]).astype(int)

In [ ]:
raw_coords = np.stack([xi, yi, zi], axis=0) + ioffset

raw_stack = ds_raw[:, :, :, T].read().result()
raw_vals = raw_stack[raw_coords[0], raw_coords[1], raw_coords[2]]

In [ ]:


# this is the aligned data
ds_aligned = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/aligned/'
}).result()


## explore a window

In [ ]:
window = 50
# (imin, jmin, T)=(949, 1033, 3629)
imin = np.random.randint(0, 2048-window)
jmin = np.random.randint(0, 1328-window)
T = np.random.randint(7800)
imax = imin + window

jmax = jmin + window

print(f"{(imin, jmin, T)=}")


vol = ds_aligned[imin:imax, jmin:jmax, :, T].read().result()

zbest = np.argmax(vol.max((0, 1)))
gx = imin // STRIDE_X
gy = jmin // STRIDE_Y
gz = zbest // STRIDE_Z

offset = ds_flow[:, gz, gy, gx, T].read().result()
ioffset = np.round(offset / [1, 1, 4]).astype(int)
print(ioffset)
rimin, rimax = np.array([imin, imax]) + ioffset[0]
rjmin, rjmax = np.array([jmin, jmax]) + ioffset[1]
rzbest = zbest + ioffset[2]

raw_vol = ds_raw[rimin:rimax, rjmin:rjmax, :, T].read().result()
_, ax = plt.subplots(1, 2)
ax[0].imshow(vol[:, :, zbest])
ax[1].imshow(raw_vol[:, :, rzbest])
plt.suptitle("Best z plane")

In [ ]:
_, ax = plt.subplots(1, 2)
ax[0].imshow(vol.max(2))
ax[1].imshow(raw_vol.max(2))
plt.suptitle("MIP")